In [1]:
#coding=utf-8
# ------------------------------------------------------------------------------------------#
# 
#-------------------------------------------------------------------------------------------#
from netCDF4 import Dataset as ncfile
from netCDF4 import num2date, date2num
import xarray as xr
from datetime import datetime, timedelta
import os, fnmatch, glob
import numpy as np
import numpy.ma as ma
import pandas as pd
import math
import matplotlib.dates as dates
from pyproj import Proj, transform
import scipy
from scipy import spatial
import cartopy.crs as ccrs
import cmocean
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

/home/maltem/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
# Find nearest neighbor point index
def do_kdtree(combined_x_y_arrays,points):
        mytree = scipy.spatial.cKDTree(combined_x_y_arrays)
        dist, indexes = mytree.query(points)
        return indexes
    
#
def remapfield(MODvar,idx_pos_ij):
    MODprojvar=np.zeros(np.shape(idx_pos_ij))
    for i in range(np.shape(idx_pos_ij)[0]):
        for j in range(np.shape(idx_pos_ij)[1]):
            MODprojvar[i][j]=MODvar[idx_pos_ij[i][j]]
    return MODprojvar

class SpatialEvaluation:
    
    def __init__(self,modeleval):
        self.model=modeleval
    
    def defineGridTransforms(self):
        
        #MODIS grid (same as AROME Arctic)      
        modisin='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/observations/remotesensing/aa-modis-2024.nc'
        AROMEin='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET_AROMEArctic/AROME_ARCTIC/2024/04/04/arome_arctic_det_2_5km_20240404T00Z.nc'
        #
        modisdataset = ncfile(modisin, 'r')     
        aromedataset = ncfile(AROMEin, 'r') # not sufficient information in modis file
        modis_lons = aromedataset.variables['longitude'][:,:]
        modis_lats = aromedataset.variables['latitude'][:,:]
        modis_x = aromedataset.variables['x'][:]
        modis_y = aromedataset.variables['y'][:]
        modis_times = modisdataset.variables['time'][:]
        t_unit = modisdataset.variables['time'].units
        modis_nctimes=[]; modis_nctimes.append(num2date(modis_times,units = t_unit,calendar = 'proleptic_gregorian'))
        modis_nctimes=np.squeeze(modis_nctimes)
        modis_nctimenum = np.squeeze(dates.date2num(modis_nctimes))        
        #
        modis_proj = Proj("+proj=lcc +lat_1=77.5 +lat_2=77.5 +lat_0=77.5 +lon_0=-25 +R=6371000")          
        modisdataset.close()
        aromedataset.close()
            
        if self.model!='MET-AROMEArctic' and self.model!='MF-AROME' and self.model!='MF-ARPEGE':
                   
            if self.model=='ECMWF-IFS': 
                # ----- IFS Grid information
                fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECMWF-IFS/'
                MODin = fc_data_path+'Arctic.ECMWF_extracted_20240401T00Z.nc'
                #
                MODdataset = ncfile(MODin, 'r')
                MOD_lons = MODdataset.variables['lon'][:]
                MOD_lats = MODdataset.variables['lat'][:]
                MOD_times = MODdataset.variables['time'][:]
                t_unit = MODdataset.variables['time'].units
                t_cal = MODdataset.variables['time'].calendar
                MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
                MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
                #
                MOD_proj = Proj(init = 'epsg:4326')
                MODdataset.close()

            if self.model=='ECMWF-AIFS':
                fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECMWF-AIFS/'
                MODin = fc_data_path+'Arctic.ECMWF_AIFS_extracted_20240401T00Z.nc'
                #
                MODdataset = ncfile(MODin, 'r')
                MOD_lons = MODdataset.variables['lon'][:]
                MOD_lats = MODdataset.variables['lat'][:]
                MOD_times = MODdataset.variables['time'][:]
                t_unit = MODdataset.variables['time'].units
                t_cal = MODdataset.variables['time'].calendar
                MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
                MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
                #
                MOD_proj = Proj(init = 'epsg:4326')
                MODdataset.close()
            
            if self.model=='DWD-ICON':
                fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/DWD-ICON/'
                MODin = fc_data_path+'region.icon_global_2024041012_T_ICE.nc'
                #
                MODdataset = ncfile(MODin, 'r')
                MOD_lons = MODdataset.variables['lon'][:]
                MOD_lats = MODdataset.variables['lat'][:]
                MOD_times = MODdataset.variables['time'][:]
                t_unit = MODdataset.variables['time'].units
                t_cal = MODdataset.variables['time'].calendar
                MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
                MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
                #
                MOD_proj = Proj(init = 'epsg:4326')
                MODdataset.close()
            
            
            MOD_llon, MOD_llat = np.meshgrid(MOD_lons,MOD_lats)
            modis_xx, modis_yy   = np.meshgrid(modis_x, modis_y)

            # Project all on MODIS grid
            MOD_xx,MOD_yy = transform(MOD_proj, modis_proj, MOD_llon, MOD_llat)

            # Vectorize and concatenate the x,y coordinates
            modis_xx_yy = np.dstack([modis_xx.ravel(), modis_yy.ravel()])[0]
            MOD_xx_yy  = np.dstack([MOD_xx.ravel(), MOD_yy.ravel()])[0]

            #idx_pos = do_kdtree(modis_xx_yy, era5_xx_yy)     #Gives a field with dimensions of era5_xx_yy
            idx_pos = do_kdtree( MOD_xx_yy, modis_xx_yy)     #Gives a field with dimensions of modis_xx_yy

            idx_pos_ij = idx_pos.reshape(np.shape(modis_xx))  #Gives a field with dimensions of modis_xx    

            self.idx_pos_ij = idx_pos_ij
        
        self.modis_lons=modis_lons
        self.modis_lats=modis_lats
        self.modis_nctimes=modis_nctimes
        self.modis_nctimenum=modis_nctimenum
        
        
    def ModelTransform(self,filein,timestep,variable):
        
        
        if self.model=='MET-AROMEArctic':
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-AROMEArctic/'
            MODin = fc_data_path+filein
            #
            #filein = f'MET-AROMEArctic/AROME_ARCTIC/{YEAR}/{MM}/{DD}/arome_arctic_det_2_5km_{formatted_date}T{HH}Z.nc'
            #variable='SFX_TS'
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = "proleptic_gregorian" 
            MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
            MOD_nctimes =np.squeeze(MOD_nctimes)   
        
        if self.model=='MF-AROME':
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MF-AROME/merged/'
            MODin = fc_data_path+filein
            #
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = MODdataset.variables['time'].calendar
            if "day as %Y%m%d.%f" in t_unit:
                    MOD_nctimes = [
                        datetime.strptime(str(int(time)), "%Y%m%d") + timedelta(days=time % 1)
                        for time in MOD_times
                    ]      
            #MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimes   = np.squeeze(MOD_nctimes)
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
    
        if self.model=='MF-ARPEGE':
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MF-ARPEGE/merged/'
            MODin = fc_data_path+filein
            #
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = MODdataset.variables['time'].calendar
            if "day as %Y%m%d.%f" in t_unit:
                    MOD_nctimes = [
                        datetime.strptime(str(int(time)), "%Y%m%d") + timedelta(days=time % 1)
                        for time in MOD_times
                    ]      
            #MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimes   = np.squeeze(MOD_nctimes)
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
            
        if self.model=='ECMWF-IFS': 
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECMWF-IFS/' 
            MODin = fc_data_path+filein
            #
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = MODdataset.variables['time'].calendar
            MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimes   = np.squeeze(MOD_nctimes)
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
            
            
        if self.model=='ECMWF-AIFS':
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECMWF-AIFS/'
            MODin = fc_data_path+filein
            #
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = MODdataset.variables['time'].calendar
            MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimes   = np.squeeze(MOD_nctimes)
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
            
            
        if self.model=='DWD-ICON':
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/DWD-ICON/'
            MODin = fc_data_path+filein
            #
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = MODdataset.variables['time'].calendar
            MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimes   = np.squeeze(MOD_nctimes)
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
            
              
        MODdataset.close()
               
        self.MODnctimes   = MOD_nctimes[timestep]
        self.MODnctimenum = MOD_nctimenum[timestep]
                
        if self.model=='MET-AROMEArctic':    
            self.MODprojdata  = np.squeeze(MODdata)
        elif self.model=='MF-AROME':
            tmpdata = np.squeeze(MODdata)
            self.MODprojdata  = tmpdata[::2, ::2]
        elif self.model=='MF-ARPEGE':
            self.MODprojdata  = np.squeeze(MODdata)
        else:
            self.MODprojdata  = remapfield(MODdata.ravel(),self.idx_pos_ij)
        
        
        
    def DefineSICMask(self):
        
        date = self.MODnctimes
        
        # LOAD AMSR2 Sea-Ice Product
        fc_data_path   = '/lustre/storeB/project/fou/hi/oper/barents_eps/archive/obs/'
        file_name_stub = 'barents_icec-obs_%s' #barents_icec-obs_20240415T00Z.nc
        datetime_str = f"{date.year:04d}{date.month:02d}{date.day:02d}"
        file_prename = file_name_stub % datetime_str
        infile_name = glob.glob(fc_data_path + file_prename + '*.nc')
        
        if infile_name:
            
            IceIn = xr.open_mfdataset(infile_name)
            lats = np.array(IceIn['lat'])
            lons = np.array(IceIn['lon'])
            sicn = np.array(IceIn['ice_conc'])
            
            #sicn=np.squeeze(np.where(sicn<0.99,np.nan,sicn))  
            
            self.SIC_mask=np.squeeze(np.flipud(np.squeeze(sicn)))
            IceIn.close()
                  
        
    def RetrieveMODIS(self,masksic):
        print ('--> RetrieveMODIS')
        #MODIS grid (same as AROME Arctic)      
        modisin='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/observations/remotesensing/aa-modis-2024.nc'
    
        modisdataset = ncfile(modisin, 'r')       
        tm = np.argmin(np.abs(self.MODnctimenum-self.modis_nctimenum))
        
        print('   Model date:',self.MODnctimes)
        print('   MODIS date:',self.modis_nctimes[tm])
            
        tempmodis = np.squeeze(modisdataset['modis_sist_lw'][tm,:,:])
        
        tempmodis = ma.filled(tempmodis, fill_value=np.nan)
        
        
        
        
        if masksic:
            print ('   Masking sic < 0.2')
            tempmodis = np.squeeze(np.where(self.SIC_mask<0.01,np.nan,tempmodis.data))
          #  tempmodis = np.squeeze(np.where(self.SIC_mask==np.nan,np.nan,tempmodis.data))
                        
        self.modisdata = tempmodis
        
        modisdataset.close()
        
    def ConditionalEvaluateModel(self,cond,cmin,cmax,cinc):
        
        if cond=='SIC':
            condfield = self.SIC_mask
            bins = np.arange(cmin, cmax + cinc, cinc) 
        elif cond=='TS':
            condfield = self.modisdata-273.15
            bins = np.arange(cmin, cmax + cinc, cinc)
            
        modfield=self.MODprojdata
        obsfield=self.modisdata
        
        bias_list=[];mse_list=[]
        # Iterate over the bins
        for i in range(len(bins) - 1):
            # Define the condition range
            lower_bound = bins[i]
            upper_bound = bins[i + 1]
        
            # Create a mask for elements within the current condition range
            condition_mask = (condfield >= lower_bound) & (condfield < upper_bound)
        
            # Extract corresponding values from modfield and obsfield
            mod_vals = modfield[condition_mask]
            obs_vals = obsfield[condition_mask]
        
            # Ensure there is data in this range before calculating
            if (len(mod_vals) > 0) and (len(obs_vals) > 0) and not np.isnan(np.nanmean(obs_vals-mod_vals)) and not np.isnan(np.nanmean(mod_vals)):
                # Calculate the bias and RMSE for this condition range
                #print ('mod mean',np.nanmean(mod_vals))
                #print ('obs mean',np.nanmean(obs_vals))
                
                bias = np.nanmean(mod_vals - obs_vals)  # Bias: mean(mod - obs)
                mse = np.nanmean((mod_vals - obs_vals) ** 2)  # MSE: mean((mod - obs)^2)
            
                # Append the results
                bias_list.append(bias)
                mse_list.append(mse)
            else:
                # If no data for this condition, append NaN
                bias_list.append(np.nan)
                mse_list.append(np.nan)
    
            # Return the results as a dictionary
        print("bias", bias_list,"mse", mse_list,"bins", bins[:-1])  # To show which bin corresponds to each result
        self.bias = np.array(bias_list)
        self.mse  = np.array(mse_list)     
        self.bins = np.array(bins)


In [ ]:
def conditionalEvaluation(start_date,end_date,start_leadtime,end_leadtime,modeleval,condvar,cmin,cmax,cinc):


    m1 = SpatialEvaluation(modeleval=modeleval)
    m1.defineGridTransforms()

    # Loop over the days from start to end
    current_date = start_date
    first=1
    while current_date <= end_date:
        # Format the date for the filename (e.g., 20240410)
        formatted_date = current_date.strftime('%Y%m%d')
        YEAR=current_date.strftime('%Y')
        MM  =current_date.strftime('%m')
        DD  =current_date.strftime('%d')
        for HH in ['00','12']:   
        
            if modeleval=='MET-AROMEArctic':
                idir='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/'
                filein = f'AROME_ARCTIC/{YEAR}/{MM}/{DD}/arome_arctic_det_2_5km_{formatted_date}T{HH}Z.nc'
                variable='SFX_TS'
            if modeleval=='MF-AROME':
                filein = f'AROME_SVALBARD_{formatted_date}{HH}00.nc'
                variable='TS'
            if modeleval=='MF-ARPEGE':
                filein = f'ARPEGE_SVALBARD_{formatted_date}{HH}00.nc'
                variable='TS'
            if modeleval=='ECMWF-IFS':     
                filein = f'Arctic.ECMWF_extracted_{formatted_date}T{HH}Z.nc'
                variable='SKT'
            if modeleval=='ECMWF-AIFS':
                filein = f'Arctic.ECMWF_AIFS_extracted_{formatted_date}T{HH}Z.nc'
                variable='skt'
            if modeleval=='DWD-ICON':
                filein = f'region.icon_global_{formatted_date}{HH}_T_ICE.nc'
                variable='ist'
            
            
            print ('Input File: ',filein)
            
            for ti in range(start_leadtime,end_leadtime+1):
    
                # Call the ModelTransform method with the current date's filename
                m1.ModelTransform(filein=filein, timestep=ti, variable=variable)
                m1.DefineSICMask()
                m1.RetrieveMODIS(masksic=True)
                m1.ConditionalEvaluateModel(cond=condvar,cmin=cmin ,cmax=cmax,cinc=cinc)
                if first==1:
                    bias=m1.bias.reshape(len(m1.bins)-1, 1)
                    mse=m1.mse.reshape(len(m1.bins)-1, 1)
                    first=0
                else:
                    bias=np.hstack((bias,m1.bias.reshape(len(m1.bins)-1, 1)))
                    mse=np.hstack((mse,m1.mse.reshape(len(m1.bins)-1, 1)))
                
            

        # Move to the next day
        current_date += timedelta(days=1)
    
    rmse_values = np.sqrt(np.nanmean(mse,axis=1))
    bias_values = np.nanmean(bias,axis=1)
    return rmse_values,bias_values


### 1 - Evaluation, conditional = SIC, leadtime 13-24h 

In [ ]:
# Define start and end date
start_date = datetime(2024, 4, 6)  # 
end_date = datetime(2024, 4, 30)    # 

pls=13;ple=25
als=2 ;ale=5
condvar = 'SIC'

models = ['ECMWF-IFS', 'ECMWF-AIFS', 'DWD-ICON', 'MET-AROMEArctic', 'MF-AROME', 'MF-ARPEGE']
rmse_values=[]; bias_values=[]
#models = ['MF-AROME']

for model in models:

    if model=='ECMWF-AIFS':
        ls=als;  le=ale
    else:
        ls=pls;  le=ple
    
    # Run conditional evaluations for each model and store results
    rmse, bias = conditionalEvaluation(start_date, end_date, ls, le, model, condvar, 0., 1., 0.2)
    rmse_values.append(rmse1)
    bias_values.append(bias1)


# Create a DataFrame to organize the results
results_df = pd.DataFrame({
    'Model': models,
    'RMSE': rmse_values,
    'Bias': bias_values
})

# Save the results to a CSV file
results_df.to_csv('MODISeval_'+condvar+'_'+str(pls)+'-'+str(ple)+'.csv', index=False)

print("Results saved ...")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

bins = np.arange(0.0, 1.2, 0.2)
bin_midpoint = bins[0:5] + 0.1


# Create subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), sharey=False)

# Left: RMSE Plot
ax1.vlines(bins, ymin=2, ymax=10, colors='skyblue', linestyles='solid', label="Bin Heights")
ax1.scatter(bin_midpoint, rmse1, color='red', label='ECMWF-IFS', marker='o')
ax1.scatter(bin_midpoint, rmse2, color='blue', label='ECMWF-AIFS', marker='*')
ax1.scatter(bin_midpoint, rmse3, color='purple', label='DWD-ICON', marker='D')
ax1.scatter(bin_midpoint, rmse4, color='orange', label='MET-AROMEArctic', marker='s')
ax1.scatter(bin_midpoint, rmse5, color='brown', label='MF-AROME', marker='p')
ax1.scatter(bin_midpoint, rmse6, color='cyan', label='MF-ARPEGE', marker='v')

ax1.set_xlabel("Sea-Ice Concentration", fontsize=12)
ax1.set_ylabel("RMSE", fontsize=12)
ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax1.legend(fontsize=10)
ax1.set_title("RMSE", fontsize=14)

# Right: BIAS Plot
ax2.vlines(bins, ymin=-3, ymax=6, colors='skyblue', linestyles='solid', label="Bin Heights")
ax2.scatter(bin_midpoint, bias1, color='red', label='ECMWF-IFS', marker='o')
ax2.scatter(bin_midpoint, bias2, color='blue', label='ECMWF-AIFS', marker='*')
ax2.scatter(bin_midpoint, bias3, color='purple', label='DWD-ICON', marker='D')
ax2.scatter(bin_midpoint, bias4, color='orange', label='MET-AROMEArctic', marker='s')
ax2.scatter(bin_midpoint, bias5, color='brown', label='MF-AROME', marker='p')
ax2.scatter(bin_midpoint, bias5, color='cyan', label='MF-ARPEGE', marker='v')

ax2.set_xlabel("Sea-Ice Concentration", fontsize=12)
ax2.set_title("BIAS", fontsize=14)
ax2.grid(axis='y', linestyle='--', alpha=0.7)
#ax2.legend(fontsize=10)

# Adjust layout and display
plt.tight_layout()
plt.show()


In [ ]:
bias1

In [ ]:
# Define start and end date
start_date = datetime(2024, 4, 5)  # 
end_date = datetime(2024, 4, 18)    # 

ls=35;le=36
cmax=2
cmin=-28
cinc=3

#[rmse1,bias1]=conditionalEvaluation(start_date,end_date,ls,le,'ECMWF-IFS','TS',cmin,cmax,cinc)
#[rmse2,bias2]=conditionalEvaluation(start_date,end_date,6,6,'ECMWF-AIFS','TS',cmin,cmax,cinc)
#[rmse3,bias3]=conditionalEvaluation(start_date,end_date,ls,le,'DWD-ICON','TS',cmin,cmax,cinc)
#[rmse4,bias4]=conditionalEvaluation(start_date,end_date,ls,le,'MET-AROMEArctic','TS',cmin,cmax,cinc)
[rmse5,bias5]=conditionalEvaluation(start_date,end_date,ls,le,'MF-AROME','TS',cmin,cmax,cinc)
[rmse6,bias6]=conditionalEvaluation(start_date,end_date,ls,le,'MF-ARPEGE','TS',cmin,cmax,cinc)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

bins = np.arange(cmin, cmax, cinc)
bin_midpoint = bins[0:len(bins)] + cinc/2.


# Create subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), sharey=False)

# Left: RMSE Plot
ax1.vlines(bins, ymin=2, ymax=8, colors='skyblue', linestyles='solid', label="Bin Heights")
ax1.scatter(bin_midpoint, rmse1, color='red', label='ECMWF-IFS', marker='o')
ax1.scatter(bin_midpoint, rmse2, color='blue', label='ECMWF-AIFS', marker='*')
ax1.scatter(bin_midpoint, rmse3, color='purple', label='DWD-ICON', marker='D')
ax1.scatter(bin_midpoint, rmse4, color='orange', label='MET-AROMEArctic', marker='s')
ax1.scatter(bin_midpoint, rmse5, color='brown', label='MF-AROME', marker='p')
ax1.scatter(bin_midpoint, rmse6, color='cyan', label='MF-ARPEGE', marker='v')

ax1.set_xlabel("Sea-Ice Concentration", fontsize=12)
ax1.set_ylabel("RMSE", fontsize=12)
ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax1.legend(fontsize=10)
ax1.set_title("RMSE", fontsize=14)

# Right: BIAS Plot
ax2.vlines(bins, ymin=-5, ymax=6, colors='skyblue', linestyles='solid', label="Bin Heights")
ax2.scatter(bin_midpoint, bias1, color='red', label='ECMWF-IFS', marker='o')
ax2.scatter(bin_midpoint, bias2, color='blue', label='ECMWF-AIFS', marker='*')
ax2.scatter(bin_midpoint, bias3, color='purple', label='DWD-ICON', marker='D')
ax2.scatter(bin_midpoint, bias4, color='orange', label='MET-AROMEArctic', marker='s')
ax2.scatter(bin_midpoint, bias5, color='brown', label='MF-AROME', marker='p')
ax2.scatter(bin_midpoint, bias5, color='cyan', label='MF-ARPEGE', marker='v')

ax2.set_xlabel("Sea-Ice Concentration", fontsize=12)
ax2.set_title("BIAS", fontsize=14)
ax2.grid(axis='y', linestyle='--', alpha=0.7)
#ax2.legend(fontsize=10)

# Adjust layout and display
plt.tight_layout()
plt.show()


In [ ]:
bin_midpoint

#### Some test environment

In [ ]:
m1 = SpatialEvaluation(modeleval='MF-ARPEGE')
m1.defineGridTransforms()

m1.ModelTransform(filein = 'ARPEGE_SVALBARD_202404150000.nc',timestep=1,variable='TS')

m1.DefineSICMask()
m1.RetrieveMODIS(masksic=True)
m1.ConditionalEvaluateModel(cond='TS',cmin=-20. ,cmax=1.0 ,cinc=2)



In [ ]:
print (np.shape(m1.MODprojdata))
print (np.shape(m1.MODprojdata[::2, ::2]))

In [ ]:
plot_proj = ccrs.Orthographic(14,25)
fig, ax = plt.subplots(figsize=(15, 7),nrows=1,ncols=3,subplot_kw={'projection': plot_proj})

ax[0].coastlines(resolution='50m')
ax[0].set_extent([-10, 30, 67, 88], crs=ccrs.PlateCarree())

contour1 = ax[0].pcolormesh(m1.modis_lons, m1.modis_lats, m1.modisdata, transform=ccrs.PlateCarree())

ax[1].coastlines(resolution='50m')
ax[1].set_extent([-10, 30, 67, 88], crs=ccrs.PlateCarree())

contour1 = ax[1].pcolormesh(m1.modis_lons, m1.modis_lats,  m1.SIC_mask, transform=ccrs.PlateCarree())

ax[2].coastlines(resolution='50m')
ax[2].set_extent([-10, 30, 67, 88], crs=ccrs.PlateCarree())

#contour1 = ax[2].pcolormesh(m1.modis_lons, m1.modis_lats, m1.MODprojdata-m1.modisdata , transform=ccrs.PlateCarree(), cmap=cmocean.cm.tarn, vmin=-10,vmax=10)
contour1 = ax[2].pcolormesh(m1.modis_lons, m1.modis_lats, m1.MODprojdata , transform=ccrs.PlateCarree())

cbar = fig.colorbar(contour1, ax=ax,location = 'right')

In [ ]:
m1 = SpatialEvaluation(modeleval='MET-AROMEArctic')
m1.defineGridTransforms()

ti=3

plot_proj = ccrs.Orthographic(14,25)
fig, ax = plt.subplots(figsize=(15, 7),nrows=1,ncols=1,subplot_kw={'projection': plot_proj})

ax.coastlines(resolution='50m')
ax.set_extent([2, 25, 77, 83.5], crs=ccrs.PlateCarree())



start_date = datetime(2024, 4, 6)  # 
end_date = datetime(2024, 4, 7)  # 
current_date = start_date
first=1
while current_date <= end_date:

    for ti in range(24):
    
        formatted_date = current_date.strftime('%Y%m%d')
        YEAR=current_date.strftime('%Y')
        MM  =current_date.strftime('%m')
        DD  =current_date.strftime('%d')
        HH  ='00'   


        idir='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/'
        filein = f'AROME_ARCTIC/{YEAR}/{MM}/{DD}/arome_arctic_det_2_5km_{formatted_date}T{HH}Z.nc'
        variable='SFX_TS'

        m1.ModelTransform(filein=filein, timestep=ti, variable=variable)
        m1.DefineSICMask()
        m1.RetrieveMODIS(masksic=True)


        contour1 = ax.pcolormesh(m1.modis_lons, m1.modis_lats,  m1.modisdata-273.15, cmap=cmocean.cm.thermal, transform=ccrs.PlateCarree())

        #cbar = fig.colorbar(contour1, ax=ax,location = 'right')
        contour1.set_clim(-40, 0)  
        plt.savefig('anim/MODIS'+formatted_date+str(ti)+'.png', dpi=300) 
    current_date += timedelta(days=1)